# Claude 토크나이저, 한국어와 영어 토큰 수 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/claude_tokenizer_korean.ipynb)

같은 내용을 담은 한영 문단을 보내고, 모델마다 기준 문자열 `a`의 입력 토큰 수를 빼서 순 토큰 수를 구합니다.
위키백과 문서의 길이를 늘려 가며 글자당 토큰 수를 재고, 같은 계산을 코드·표·문자군에도 적용합니다.
한영 문단에서 구한 글자당 토큰 수로 100만 자를 입력할 때의 요금을 추정합니다.

`count_tokens` 호출은 무료이며 Anthropic API 키가 필요합니다. Colab의 열쇠 아이콘에서 `ANTHROPIC_API_KEY`를 등록하고 이 노트북에 접근을 허용하세요. [Anthropic 토큰 계수 안내](https://platform.claude.com/docs/en/build-with-claude/token-counting)

| 기본 비교 그룹 | 모델 ID | 기본 실험 |
|---|---|---|
| 구 모델 | `claude-haiku-4-5` | A · B · C · D · E |
| 구 모델 | `claude-sonnet-4-6` | A · C · D · E |
| 신 모델 | `claude-haiku-5-5` | A · B · C · D · E |
| 신 모델 | `claude-sonnet-5-5` | A · C · D · E |
| 신 모델 | `claude-opus-4-7` | A · C |
| 신 모델 | `claude-opus-5-5` | A · C |

아래 설정에서 `MODELS`를 바꾸면 각 그룹에서 먼저 나오는 모델끼리 비교합니다. `QUICK = True`로 설정하면 적은 예문으로 전체 과정을 한 번씩 실행합니다.

In [ ]:
%pip install -q anthropic

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from getpass import getpass
import hashlib
import json
import math
import os
from pathlib import Path
import random
from statistics import median
import string
import threading
import time
import unicodedata
from urllib.parse import quote
import warnings

import anthropic
import requests

QUICK = False
CHART_LANG = os.environ.get("CHART_LANG", "ko")
OLD = [
    "claude-haiku-4-5", "claude-sonnet-4-6",
]
NEW = [
    "claude-haiku-5-5", "claude-sonnet-5-5",
    "claude-opus-4-7", "claude-opus-5-5",
]
MODELS = OLD + NEW
REPEATS = 1 if QUICK else 3
KRW_PER_USD = 1400
# 입력 100만 토큰당 USD. Haiku 5.5는 요청당 10만 토큰 이하.
PRICE = {OLD[0]: 1, NEW[0]: .10, OLD[1]: 3, NEW[1]: 2}
TOPICS = [
    ("한글", "Hangul"), ("광합성", "Photosynthesis"),
    ("서울특별시", "Seoul"), ("컴퓨터", "Computer"),
    ("조선", "Joseon"),
]
LIMITS = (1000, 4000, 16000, 64000)
CACHE = Path("cache")
RESULTS = Path("results")
FONT_URL = (
    "https://github.com/google/fonts/raw/main/"
    "ofl/nanumgothic/NanumGothic-Regular.ttf"
)
PALETTE = ("#426984", "#986348", "#444444")

if CHART_LANG not in ("ko", "en", "both"):
    raise ValueError("CHART_LANG은 ko, en, both 중 하나입니다.")
if not math.isfinite(KRW_PER_USD) or KRW_PER_USD <= 0:
    raise ValueError("환율은 양수여야 합니다.")
models = list(dict.fromkeys(MODELS))
if not models:
    raise ValueError("측정할 모델을 지정하세요.")
unknown = iter(m for m in models if m not in OLD + NEW)
pair = []
for group in (OLD, NEW):
    first = next((m for m in models if m in group), None)
    pair.append(first or next(unknown, None))
subset = [m for m in pair if m]
probe_models = subset.copy()
for group in (OLD, NEW):
    others = (m for m in models if m in group and m not in pair)
    extra = next(others, None)
    if extra:
        probe_models.append(extra)

os.environ.setdefault("MPLCONFIGDIR", str(CACHE / "matplotlib"))
import matplotlib.pyplot as plt
from matplotlib import font_manager

In [ ]:
def configure_key():
    name = "ANTHROPIC_API_KEY"
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
    source = "userdata"
    if not key:
        key = os.environ.get(name)
        source = "env"
    if not key:
        key = getpass("Anthropic API 키 입력: ")
        source = "getpass"
    if not key:
        raise ValueError("빈 API 키입니다.")
    os.environ[name] = key
    print(source)


configure_key()

## 직접 작성한 한영 문단과 혼합 형식 예문

여덟 장르에 대해 같은 내용의 한국어·영어 문단을 한 쌍씩 준비했습니다. 형식 비교에 쓰는 여덟 예문은 들여쓰기, 공백, 줄바꿈까지 그대로 측정합니다.

In [ ]:
# 직접 작성한 예문이므로 재배포에 따른 라이선스 문제가 없습니다.
parallel = [
    {
        "genre_ko": "뉴스 기사",
        "genre_en": "News report",
        "ko": (
            "마을 도서관이 다음 달부터 토요일 열람 시간을 두 "
            "시간 늘린다. 평일에 도서관을 찾기 어려운 주민들이 "
            "지난봄부터 연장을 요청한 데 따른 결정이다. 시범 "
            "운영은 석 달 동안 이어지며, 자료 대출과 반납도 "
            "같은 시간까지 가능하다. 도서관은 매주 시간대별 "
            "방문 인원을 기록하고 이용자 의견을 모은 뒤, "
            "가을에 연장 운영을 계속할지 결정할 예정이다."
        ),
        "en": (
            "The village library will stay open two hours "
            "later on Saturdays starting next month. Residents "
            "who find it hard to visit on weekdays have been "
            "asking for longer hours since last spring. The "
            "trial will run for three months, with loans and "
            "returns available until closing time. The library "
            "will track weekly visits by time of day and "
            "collect feedback before deciding in the fall "
            "whether to keep the longer hours."
        ),
    },
    {
        "genre_ko": "고객 문의",
        "genre_en": "Customer inquiry",
        "ko": (
            "지난주에 주문한 접이식 선반을 오늘 받아 조립해 "
            "보았습니다. 설명서에는 짧은 나사가 여섯 개 들어 "
            "있다고 적혀 있는데, 포장을 모두 펼쳐 보아도 다섯 "
            "개뿐입니다. 긴 나사로 대신 고정하면 나무판을 뚫고 "
            "나올 것 같아 조립을 멈췄습니다. 선반 전체를 다시 "
            "포장하기가 어려운데, 빠진 나사만 따로 받아 볼 수 "
            "있을까요? 확인에 필요한 사진도 보내 드리겠습니다."
        ),
        "en": (
            "The folding shelf I ordered last week arrived "
            "today, and I started putting it together. The "
            "instructions list six short screws, but I could "
            "only find five after unpacking everything. I "
            "stopped because a long screw looks like it would "
            "go straight through the board. It would be "
            "difficult to pack up the whole shelf again. Could "
            "you send just the missing screw? I can send "
            "photos if that helps."
        ),
    },
    {
        "genre_ko": "기술 문서",
        "genre_en": "Technical manual",
        "ko": (
            "측정 기록을 내보내려면 먼저 장치의 시계가 올바른 "
            "시간대로 설정되어 있는지 확인한다. 기록 화면에서 "
            "시작 날짜와 종료 날짜를 선택한 뒤 저장 단추를 "
            "누르면 날짜순으로 정렬된 파일이 생성된다. 비어 "
            "있는 측정값은 숫자 영과 구분하기 위해 빈칸으로 "
            "남는다. 파일을 다른 프로그램에서 열 때에는 첫 "
            "줄을 열 이름으로 읽도록 설정한다."
        ),
        "en": (
            "Before exporting measurements, check that the "
            "device clock is set to the correct time zone. "
            "Select the start and end dates on the records "
            "screen and press Save to create a file sorted by "
            "date. Missing readings are left blank to "
            "distinguish them from zero. When opening the file "
            "in another program, set it to read the first row "
            "as column names."
        ),
    },
    {
        "genre_ko": "약관 조항",
        "genre_en": "Terms of use",
        "ko": (
            "공동 작업실 이용자는 예약한 시간 안에 사용한 "
            "도구를 제자리에 두고 작업대를 정리해야 한다. "
            "장비에 이상이 발견되면 즉시 사용을 멈추고 관리 "
            "담당자에게 알려야 한다. 앞선 이용자의 정리 "
            "지연으로 작업을 시작하지 못한 경우에는 그 시간을 "
            "이용 기록에서 제외한다. 다만 다음 예약이 있으면 "
            "종료 시각은 그대로 유지하며, 차감된 시간은 다음 "
            "방문 때 사용할 수 있다."
        ),
        "en": (
            "Shared workshop users must return tools and clear "
            "their benches within their reserved time. Anyone "
            "who discovers a problem with the equipment must "
            "stop using it immediately and notify the "
            "attendant. If the previous user is late clearing "
            "up, the resulting delay is deducted from the new "
            "user's recorded time. If another booking follows, "
            "the end time stays the same and the deducted time "
            "may be used on the next visit."
        ),
    },
    {
        "genre_ko": "일상 대화",
        "genre_en": "Everyday conversation",
        "ko": (
            "“내일 산책할 때는 다리 건너 작은 길로 가 볼까? "
            "지난번에는 큰길로만 돌아서 물가에 핀 꽃을 제대로 "
            "못 봤잖아.” “좋아. 그런데 아침에 비가 오면 흙길이 "
            "미끄러울 텐데, 점심 먹고 출발하는 게 어때?” "
            "“그러자. 물은 내가 챙길게. 돌아오는 길에 앉아서 "
            "쉴 만한 자리를 찾으면 거기서 간식도 나눠 먹자.” "
            "“그럼 나는 사과 두 개를 씻어서 가져갈게.”"
        ),
        "en": (
            "“Shall we take the little path across the bridge "
            "on our walk tomorrow? Last time we stuck to the "
            "main road and barely saw the flowers by the "
            "water.” “Sure. If it rains in the morning, "
            "though, the dirt path will be slippery. How about "
            "leaving after lunch?” “Let's do that. I'll bring "
            "water. If we find a place to sit on the way back, "
            "we can stop and share a snack.” “Then I'll wash a "
            "couple of apples and bring them along.”"
        ),
    },
    {
        "genre_ko": "제품 설명",
        "genre_en": "Product description",
        "ko": (
            "이 천 가방은 입구를 넓게 벌릴 수 있어 장을 본 뒤 "
            "채소나 빵을 눕혀 담기 좋다. 안쪽의 작은 "
            "주머니에는 열쇠처럼 바닥에 묻히기 쉬운 물건을 "
            "따로 넣을 수 있다. 손잡이는 두 겹으로 덧대어 손에 "
            "닿는 부분이 쉽게 접히지 않는다. 쓰지 않을 때에는 "
            "바닥의 솔기를 따라 납작하게 접어 서랍에 보관하고, "
            "세탁한 뒤에는 모양을 펴서 그늘에 말린다."
        ),
        "en": (
            "This cloth bag opens wide so you can lay "
            "vegetables or bread flat after shopping. A small "
            "inside pocket keeps things like keys from getting "
            "buried at the bottom. Two layers of fabric help "
            "the handles keep their shape where you grip them. "
            "When the bag is empty, fold it flat along the "
            "bottom seam and store it in a drawer. After "
            "washing, reshape it and let it dry in the shade."
        ),
    },
    {
        "genre_ko": "요리법",
        "genre_en": "Recipe",
        "ko": (
            "감자 두 개를 얇게 썰어 찬물에 잠시 담가 둔다. "
            "그동안 양파 반 개를 잘게 다지고 냄비에 기름을 "
            "조금 둘러 투명해질 때까지 볶는다. 감자는 물기를 "
            "빼고 양파와 함께 한 번 뒤적인 다음, 재료가 겨우 "
            "잠길 만큼 물을 붓는다. 감자가 숟가락으로 쉽게 "
            "으깨질 정도로 익으면 불을 줄인다. 우유를 조금씩 "
            "부어 원하는 농도를 맞추고 마지막에 소금으로 "
            "간한다."
        ),
        "en": (
            "Thinly slice two potatoes and soak them briefly "
            "in cold water. Meanwhile, finely chop half an "
            "onion and cook it in a little oil in a pot until "
            "translucent. Drain the potatoes, toss them with "
            "the onion, and add just enough water to cover. "
            "When the potatoes mash easily with a spoon, turn "
            "down the heat. Stir in milk a little at a time "
            "until the soup is as thick as you like, then "
            "season with salt."
        ),
    },
    {
        "genre_ko": "논문 초록",
        "genre_en": "Research abstract",
        "ko": (
            "본 연구는 같은 내용을 담은 안내문에서 문장 길이가 "
            "독자의 과제 수행에 미치는 영향을 살펴보았다. "
            "참가자는 짧은 문장 또는 긴 문장으로 작성된 "
            "안내문을 읽고 제시된 순서에 따라 종이를 접었다. "
            "두 집단의 완성 시간에는 뚜렷한 차이가 없었으나, "
            "짧은 문장을 읽은 집단은 앞 단계를 다시 확인한 "
            "횟수가 적었다. 이 결과는 안내문의 난도를 평가할 "
            "때 읽기 시간과 함께 확인 행동도 살펴볼 필요가 "
            "있음을 시사한다."
        ),
        "en": (
            "This study examined how sentence length affects "
            "task performance when instructions convey the "
            "same information. Participants read instructions "
            "written in short or long sentences and folded "
            "paper in the specified order. Completion times "
            "did not differ markedly between the two groups, "
            "but readers of short sentences checked earlier "
            "steps less often. The findings suggest that "
            "evaluations of instructional difficulty should "
            "consider checking behavior alongside reading time."
        ),
    },
]

formats = [
    {
        "name_ko": "한글 키 JSON",
        "name_en": "Korean-key JSON",
        "text": (
            "{\n"
            "  \"기록명\": \"공동 온실 관찰 일지\",\n"
            "  \"측정 단위\": {\n"
            "    \"기온\": \"섭씨\",\n"
            "    \"습도\": \"백분율\"\n"
            "  },\n"
            "  \"관찰 기록\": [\n"
            "    {\n"
            "      \"구역\": \"동쪽\",\n"
            "      \"기온\": 21.4,\n"
            "      \"습도\": 68,\n"
            "      \"환기 필요\": false,\n"
            "      \"메모\": \"새잎 두 장이 펼쳐졌다.\"\n"
            "    },\n"
            "    {\n"
            "      \"구역\": \"서쪽\",\n"
            "      \"기온\": 24.1,\n"
            "      \"습도\": 76,\n"
            "      \"환기 필요\": true,\n"
            "      \"메모\": \"창가의 흙이 아직 촉촉하다.\"\n"
            "    },\n"
            "    {\n"
            "      \"구역\": \"중앙\",\n"
            "      \"기온\": 22.8,\n"
            "      \"습도\": 71,\n"
            "      \"환기 필요\": false,\n"
            "      \"메모\": \"물받이를 비우고 잎 뒷면을 "
            "확인했다.\"\n"
            "    }\n"
            "  ],\n"
            "  \"다음 작업\": [\n"
            "    \"창문 잠금 확인\",\n"
            "    \"물통 채우기\"\n"
            "  ]\n"
            "}"
        ),
    },
    {
        "name_ko": "한글 주석 파이썬",
        "name_en": "Korean-comment Python",
        "text": (
            "def daily_means(records):\n"
            "    # 센서가 꺼져 있던 시각에는 값이 없으므로 "
            "평균에서 제외한다.\n"
            "    totals = {}\n"
            "    for day, value in records:\n"
            "        if value is None:\n"
            "            continue\n"
            "        total, count = totals.get(day, (0, 0))\n"
            "        totals[day] = (total + value, count + 1)\n"
            "\n"
            "    # 날짜순으로 내보내야 서로 다른 장치의 기록을 "
            "쉽게 비교할 수 있다.\n"
            "    return [(day, total / count)\n"
            "            for day, (total, count) in "
            "sorted(totals.items())]"
        ),
    },
    {
        "name_ko": "영문 파이썬",
        "name_en": "English Python",
        "text": (
            "def pack_rows(rows, limit):\n"
            "    \"\"\"Yield batches whose combined size fits "
            "the requested limit.\"\"\"\n"
            "    batch = []\n"
            "    size = 0\n"
            "    for row in rows:\n"
            "        if batch and size + len(row) > limit:\n"
            "            yield batch\n"
            "            batch = []\n"
            "            size = 0\n"
            "        batch.append(row)\n"
            "        size += len(row)\n"
            "    if batch:\n"
            "        yield batch"
        ),
    },
    {
        "name_ko": "이모지 채팅",
        "name_en": "Emoji chat",
        "text": (
            "🌱 오늘 모종 옮겨 심은 사람 있어? 작은 화분 "
            "밑으로 뿌리가 보여서 이번 주말까지 기다려도 될지 "
            "모르겠어.\n"
            "🪴 나는 어제 옮겼어! 새 화분에 흙을 조금 깔고 "
            "통째로 들어 올리니까 생각보다 금방 끝나더라.\n"
            "🌱 고마워 😊 물은 옮기자마자 줬어? 지난번에는 "
            "물이 아래로 다 빠져나와서 바닥 닦느라 한참 "
            "걸렸거든.\n"
            "💧 받침을 먼저 놓으면 편해. 나는 물을 두 번에 "
            "나눠 주고 십 분쯤 뒤에 받침만 비웠어.\n"
            "🪴 맞아, 그리고 첫날은 햇빛이 너무 센 자리만 "
            "피하면 돼. 내일 아침 잎이 어떻게 펴졌는지 사진도 "
            "보여 줘! 📷\n"
            "🌱 좋아! 빈 화분이랑 받침부터 꺼내 놓을게. "
            "이번에는 바닥도 무사하겠지 😄"
        ),
    },
    {
        "name_ko": "숫자 CSV",
        "name_en": "Numeric CSV",
        "text": (
            "day,hour,room,temp_c,humidity,volume_ml\n"
            "04-01,08,01,20.0,60,120.0\n"
            "04-01,09,02,21.1,61,123.0\n"
            "04-01,10,03,22.2,62,126.0\n"
            "04-01,11,01,23.3,63,129.0\n"
            "04-02,08,02,24.4,64,132.0\n"
            "04-02,09,03,20.5,65,135.0\n"
            "04-02,10,01,21.6,66,138.0\n"
            "04-02,11,02,22.0,67,141.0\n"
            "04-03,08,03,23.1,68,144.0\n"
            "04-03,09,01,24.2,69,147.0\n"
            "04-03,10,02,20.3,70,150.0\n"
            "04-03,11,03,21.4,71,153.0\n"
            "04-04,08,01,22.5,72,156.0\n"
            "04-04,09,02,23.6,60,159.0\n"
            "04-04,10,03,24.0,61,162.0\n"
            "04-04,11,01,20.1,62,165.0\n"
            "04-05,08,02,21.2,63,168.0\n"
            "04-05,09,03,22.3,64,171.0"
        ),
    },
    {
        "name_ko": "마크다운 표",
        "name_en": "Markdown table",
        "text": (
            "| 구역 | 관찰 대상 | 점검 결과 | 다음 작업 |\n"
            "|---|---|---|---|\n"
            "| 동쪽 창가 | 낮은 선반 | 오전에 빛이 오래 "
            "머무르고 잎이 한쪽으로 기울었다. | 화분 방향을 "
            "조금 돌린다. |\n"
            "| 서쪽 벽면 | 높은 선반 | 맨 아래 칸에는 바람이 "
            "거의 닿지 않았다. | 화분 사이 간격을 넓힌다. |\n"
            "| 안쪽 통로 | 물받이 | 어제 준 물이 아직 남아 "
            "있고 흙 표면도 젖어 있다. | 받침을 비우고 하루 뒤 "
            "다시 확인한다. |\n"
            "| 출입구 옆 | 어린 모종 | 문을 열 때마다 찬바람에 "
            "줄기가 흔들렸다. | 안쪽의 빈 자리로 옮긴다. |\n"
            "| 작업대 위 | 씨앗 상자 | 뚜껑 안쪽에 작은 "
            "물방울이 맺혔다. | 잠깐 열어 습기를 내보낸다. |"
        ),
    },
    {
        "name_ko": "HTML 조각",
        "name_en": "HTML fragment",
        "text": (
            "<section lang=\"ko\" "
            "aria-labelledby=\"walk-heading\">\n"
            "  <h2 id=\"walk-heading\">냇가 산책길 안내</h2>\n"
            "  <p>나무다리를 건너면 길이 두 갈래로 나뉩니다. "
            "왼쪽 길은 물가를 따라 이어지고 오른쪽 길은 작은 "
            "언덕을 돌아옵니다.</p>\n"
            "  <ul>\n"
            "    <li>비가 온 다음 날에는 돌계단이 미끄러울 수 "
            "있습니다.</li>\n"
            "    <li>벤치 옆 물통은 꽃밭에 물을 주기 위해 "
            "놓아둔 것입니다.</li>\n"
            "  </ul>\n"
            "  <footer>해가 지기 전에 입구로 돌아올 수 있도록 "
            "남은 거리를 확인해 주세요.</footer>\n"
            "</section>"
        ),
    },
    {
        "name_ko": "한글 별칭 SQL",
        "name_en": "Korean-alias SQL",
        "text": (
            "WITH daily_readings AS (\n"
            "    SELECT room_id, recorded_day,\n"
            "           AVG(temperature) AS mean_temp,\n"
            "           COUNT(*) AS sample_count\n"
            "    FROM greenhouse_readings\n"
            "    WHERE temperature IS NOT NULL\n"
            "    GROUP BY room_id, recorded_day\n"
            ")\n"
            "SELECT room_id AS \"온실 구역\",\n"
            "       recorded_day AS \"관찰 날짜\",\n"
            "       ROUND(mean_temp, 1) AS \"평균 기온\",\n"
            "       sample_count AS \"유효 측정 횟수\"\n"
            "FROM daily_readings\n"
            "ORDER BY recorded_day, room_id;"
        ),
    },
]

## 위키백과 문서 가져오기

실험 B에서 한국어와 영어 문서의 도입부부터 본문을 잘라 씁니다. 각 언어판의 MediaWiki API로 일반 텍스트를 받아 `cache/wiki/`에 저장하고, 다시 실행하면 같은 파일을 읽습니다. 각 문서의 제목·출처 URL·라이선스는 측정 결과에도 기록합니다.

출처는 [한국어 위키백과](https://ko.wikipedia.org/)와 [영어 위키백과](https://en.wikipedia.org/)의 각 문서 기여자이며, 본문 라이선스는 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)입니다.

In [ ]:
def wiki(lang, title):
    site = f"https://{lang}.wikipedia.org"
    path = CACHE / "wiki" / f"{lang}-{title}.json"
    if path.exists():
        page = json.loads(path.read_text(encoding="utf8"))
    else:
        response = requests.get(
            site + "/w/api.php",
            params=dict(
                action="query", prop="extracts", explaintext=1,
                format="json", redirects=1, titles=title,
            ),
            headers={"User-Agent": "tokenizer-study/0.1"},
            timeout=40,
        )
        response.raise_for_status()
        pages = response.json()["query"]["pages"]
        page = next(iter(pages.values()))
        if not page.get("extract", "").strip():
            raise ValueError("위키백과 본문 없음")
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(page, ensure_ascii=False),
                       encoding="utf8")
        tmp.replace(path)
    source = dict(url=site + "/wiki/" + quote(title),
                  title=page["title"], license="CC BY-SA 4.0")
    return page["extract"], source

## 측정 기준

한 번에 사용자 메시지 하나를 보내고 반복 호출의 중앙값을 사용합니다. 표의 `net`은 이 중앙값에서 같은 모델의 `count_tokens("a")` 값을 뺀 값입니다. 표의 모델 기호 M1, M2 등은 아래 출력 순서에 따릅니다. 측정값이 빠지면 표에 ‘—’로 표시합니다.

In [ ]:
def width(text):
    return sum(2 if unicodedata.east_asian_width(c) in "WF"
               else 1 for c in text)


def emit(message):
    line = ""
    for ch in str(message):
        if ch == "\n" or width(line + ch) > 64:
            print(line)
            line = ""
        if ch != "\n":
            line += ch
    print(line)


def table(headers, rows):
    cells = [headers] + [list(map(str, row)) for row in rows]
    sizes = [max(map(width, col)) for col in zip(*cells)]
    for row in cells:
        parts = [c + " " * (n - width(c))
                 for c, n in zip(row, sizes)]
        emit(" ".join(parts).rstrip())


def count(model, text):
    for attempt in range(7):
        try:
            result = client.messages.count_tokens(
                model=model,
                messages=[{"role": "user", "content": text}],
            )
            return result.input_tokens
        except anthropic.APIStatusError as exc:
            if exc.status_code != 429 and exc.status_code < 500:
                raise
            if attempt == 6:
                raise
        except anthropic.APIConnectionError:
            if attempt == 6:
                raise
        time.sleep(2 ** attempt + random.random())


def item(exp, ko, en, text, **extra):
    return dict(experiment=exp, item=ko, item_en=en,
                text=text, **extra)


def measure(task):
    job, model = task
    text = job["text"]
    data = text.encode()
    rec = {k: v for k, v in job.items() if k != "text"}
    rec.update(model=model, chars=len(text), bytes=len(data),
               sha256=hashlib.sha256(data).hexdigest(),
               counts=[], net=None)
    try:
        with lock:
            if model in missing:
                rec["skipped"] = "모델 없음"
                return rec
        for _ in range(REPEATS):
            rec["counts"].append(count(model, text))
        raw = rec["count"] = median(rec["counts"])
        if context_check(job, model) and raw > 200000:
            rec["skipped"] = "200K 초과"
            return rec
        rec["net"] = raw - bases.get(model, raw)
        if job["experiment"] == "A":
            doubled = text + "\n\n" + text
            counts = rec["double_counts"] = []
            for _ in range(REPEATS):
                counts.append(count(model, doubled))
            step = median(counts) - raw
            rec["double_increment"] = step
            rec["boundary_delta"] = step - rec["net"]
    except anthropic.NotFoundError:
        with lock:
            missing.add(model)
        rec["skipped"] = "모델 없음"
    except Exception as exc:
        status = getattr(exc, "status_code", "")
        rec["error"] = f"{type(exc).__name__} {status}"
    return rec


def context_check(job, model):
    return (job["experiment"] == "B" and model == OLD[0]
            and len(job["text"]) == 64000)


def run_jobs(exp, jobs, selected):
    tasks = [(j, m) for j in jobs for m in selected
             if exp == "base" or m in bases]
    records = []
    over_context = set()

    def collect(tasks):
        for r in pool.map(measure, tasks):
            records.append(r)
            label = f"{exp} {r['item']} {r['model']}"
            if r.get("skipped") == "200K 초과":
                over_context.add((r["item"], r["lang"]))
                emit(label + " 200K 초과, 해당 길이 제외")
            for field in ("counts", "double_counts"):
                if len(set(r.get(field, []))) > 1:
                    emit(f"반복 불일치 {label} {field} {r[field]}")

    with ThreadPoolExecutor(max_workers=8) as pool:
        collect((j, m) for j, m in tasks if context_check(j, m))
        collect((j, m) for j, m in tasks
                if not context_check(j, m)
                and not (exp == "B" and len(j["text"]) == 64000
                         and (j["item"], j["lang"])
                         in over_context))
    by_exp[exp] = records
    return records


def change(old, new):
    if old in (None, 0) or new is None:
        return None
    return 100 * (new / old - 1)


def num(n, digits=3):
    return "—" if n is None else f"{n:.{digits}f}"


def report_jobs(exp, title, jobs, selected):
    records = run_jobs(exp, jobs, selected)
    lookup = {
        (r["item"], r.get("lang"),
         r.get("limit"), r["model"]): r
        for r in records if r["net"] is not None
    }
    emit(f"\n{exp}. {title}")
    for j in jobs:
        lang, limit = j.get("lang"), j.get("limit")
        rows = {m: lookup.get((j["item"], lang, limit, m), {})
                for m in selected}
        old, new = (rows.get(m, {}).get("net") for m in pair)
        pct = change(old, new)
        length = len(j["text"])
        suffix = f" {lang}" if lang else ""
        if limit:
            suffix += f" 목표 {limit}자"
        emit(f"{j['item']}{suffix}  비교 증감 {num(pct, 2)}%")
        headers = ["모델", "글자", "net", "net/자"]
        if exp == "A":
            headers.insert(2, "바이트")
            headers.extend(["반복문 증분", "차이"])
        values = []
        for model, r in rows.items():
            rate = r["net"] / length if r else None
            row = [alias[model], length,
                   r.get("net", "—"), num(rate)]
            if exp == "A":
                row.insert(2, len(j["text"].encode()))
                row.extend([r.get("double_increment", "—"),
                            r.get("boundary_delta", "—")])
            values.append(row)
        table(headers, values)
        yield j, rows, old, new


stamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
emit(f"실행 UTC {stamp}\nSDK anthropic {anthropic.__version__}")
emit("모드 실제 API")
alias = {m: f"M{i}" for i, m in enumerate(models, 1)}
for m in models:
    emit(f"{alias[m]} {m}")
comparison = " → ".join(m or "—" for m in pair)
emit(f"반복 {REPEATS}회\n비교 {comparison}")
client = anthropic.Anthropic(max_retries=0)
missing = set()
lock = threading.Lock()
by_exp = {}
bases = {}
baseline = item("base", "기준", "Baseline", "a")
for r in run_jobs("base", [baseline], models):
    if r["net"] is not None:
        bases[r["model"]] = r["count"]
emit("기준 문자열 a")
table(["모델", "count(a)"],
      [(alias[m], bases.get(m, "—")) for m in models])

## A. 같은 내용의 한영 문단

장르마다 한영 문단을 각각 측정합니다. 문단 두 개를 빈 줄로 이어 붙였을 때의 증분을 단일 문단의 순 토큰 수와 비교합니다. 언어별 합계는 모든 장르의 측정값이 있을 때만 계산합니다.

In [ ]:
a_jobs = [
    item("A", r["genre_ko"], r["genre_en"], r[lang], lang=lang)
    for r in parallel[:2 if QUICK else None]
    for lang in ("ko", "en")
]
totals = {}
for j, rows, _, _ in report_jobs("A", "한영 문단", a_jobs, models):
    for m, r in rows.items():
        entries = totals.setdefault((j["lang"], m), [])
        entries.append(r.get("net"))

sums, char_totals = {}, {}
summary = []
for (lang, m), values in totals.items():
    texts = [j["text"] for j in a_jobs if j["lang"] == lang]
    chars = char_totals[lang] = sum(map(len, texts))
    size = sum(len(t.encode()) for t in texts)
    net = sum(values) if None not in values else None
    sums[lang, m] = net
    rate = net / chars if net is not None else None
    summary.append((lang, alias[m], chars, size,
                    num(net, 0), num(rate)))
emit("언어 합계")
table(["언어", "모델", "글자", "바이트", "net", "net/자"], summary)

language_bars = []
for lang, ko, en in [("ko", "한국어 합계", "Korean total"),
                     ("en", "영어 합계", "English total")]:
    pct = change(*(sums.get((lang, m)) for m in pair))
    language_bars.append((ko, en, pct))
    emit(f"{ko} 비교 증감 {num(pct, 2)}%")
for m in dict.fromkeys(pair):
    ko, en = (sums.get((lang, m)) for lang in ("ko", "en"))
    ratio = ko / en if ko is not None and en else None
    emit(f"{alias.get(m, '—')} 한/영 토큰 비율 {num(ratio)}")

## B. 위키백과 문서 길이

같은 주제의 두 언어판에서 앞부분을 1천, 4천, 1만 6천, 6만 4천 자까지 늘립니다. 본문이 짧으면 마지막 조각의 실제 길이를 사용합니다. 6만 4천 자에서 Haiku 4.5의 원시 계수가 20만 토큰을 넘으면 해당 문서의 그 길이를 비교에서 제외합니다. 축소 실행은 첫 주제의 첫 길이만 측정합니다.

In [ ]:
b_jobs, sources, wiki_failures = [], [], []
for titles in TOPICS[:1 if QUICK else None]:
    for lang, title in zip(("ko", "en"), titles):
        try:
            doc, source = wiki(lang, title)
            if not doc:
                raise ValueError("빈 문서")
        except (OSError, ValueError, KeyError, StopIteration,
                requests.RequestException) as exc:
            reason = type(exc).__name__
            wiki_failures.append(f"B {title} {reason}")
            continue
        sources.append(source)
        emit(source["url"])
        emit(source["license"])
        for limit in LIMITS[:1 if QUICK else None]:
            job = item("B", title, titles[1], doc[:limit],
                       lang=lang, limit=limit)
            b_jobs.append(job)
            if len(doc) <= limit:
                break

lengths = {}
for j, _, old, new in report_jobs("B", "문서 길이", b_jobs, subset):
    group = lengths.setdefault((j["lang"], j["limit"]), [])
    group.append((len(j["text"]), old, new))

## C. 혼합 형식

JSON, 파이썬, 이모지 대화, CSV, 마크다운, HTML, SQL의 예문을 모델마다 같은 형태로 보냅니다. 비교 증감은 구 모델의 순 토큰 수를 기준으로 한 백분율입니다. 축소 실행은 한글 키 JSON과 한글 주석 파이썬을 사용합니다.

In [ ]:
c_jobs = [item("C", r["name_ko"], r["name_en"], r["text"])
          for r in formats[:2 if QUICK else None]]
format_bars = []
for j, _, old, new in report_jobs("C", "혼합 형식", c_jobs, models):
    pct = change(old, new)
    format_bars.append((j["item"], j["item_en"], pct))

## D. 문자군별 글자당 순 토큰

고정 난수 시드로 문자군별 2천 자를 만들고, 한국어 문단은 실험 A의 여덟 문단을 그대로 이어 붙입니다. 그룹마다 앞의 두 모델까지 측정하고 그래프에는 대표 비교 쌍을 표시합니다. 축소 실행은 상용 한글, 전체 한글, 호환 자모만 측정합니다. 공백만 담긴 입력에서 HTTP 400이 발생하면 마지막 셀에 해당 호출이 표시됩니다.

In [ ]:
WORDS = """
the a an this that these those some each all
one two three four five six seven eight nine ten
day week month year morning evening night noon hour minute
mother father sister brother child friend family parent baby
teacher
house room kitchen door window floor wall roof garden yard
water milk bread rice salt sugar apple orange egg cheese
river lake sea hill field tree leaf flower grass root
bird fish cat dog horse sheep cow pig duck mouse
red blue green white black yellow brown gray pink purple
hand foot head face eye ear mouth nose arm leg
read write walk run sit stand talk hear look see
bring carry hold push pull turn move lift open close
make take give get keep find choose use need want
early late soon now then here there always often never
small large tall short long wide deep thin thick narrow
good bad new old young full empty warm cool cold
book page story paper pen pencil letter note word line
road street town city village school store farm park bridge
car train bus boat wheel seat bike truck plane ship
rain snow wind cloud storm sun moon star sky air
shirt coat dress shoe hat sock belt scarf pocket glove
chair table bed desk shelf lamp clock bowl cup plate
work play rest sleep eat drink cook wash clean draw
in on at by for from with to of under
and or but if so as while when because until
happy sad calm quiet loud kind brave proud tired ready
first last next left right high low near far middle
money price cost change cash bank bill coin card sale
game song music dance art film map picture photo color
time place way thing part name number point end start
""".split()


def ks_chars(start, end):
    chars = []
    for cp in range(start, end + 1):
        ch = chr(cp)
        try:
            if len(ch.encode("euc_kr")) == 2:
                chars.append(ch)
        except UnicodeEncodeError:
            pass
    return chars


def range_chars(start, end):
    return "".join(map(chr, range(start, end + 1)))


def probes(parallel):
    rng = random.Random(42)

    def sample(pool):
        return "".join(rng.choices(pool, k=2000))

    def chunks(pool, separator):
        pieces, length = [], 0
        while length < 2000:
            word = rng.choice(pool)
            length += len(word)
            if pieces:
                length += len(separator)
            pieces.append(word)
        return separator.join(pieces)[:2000]

    common = ks_chars(0xac00, 0xd7a3)
    assert len(common) == 2350
    assert len(set(WORDS)) == len(WORDS) == 300
    return [
        ("상용 한글", "Common Hangul", sample(common)),
        ("전체 한글", "All Hangul",
         sample(range_chars(0xac00, 0xd7a3))),
        ("호환 자모", "Compatibility jamo",
         sample(range_chars(0x3131, 0x318e))),
        ("한자", "Hanja", sample(ks_chars(0x4e00, 0x9fff))),
        ("히라가나", "Hiragana",
         sample(range_chars(0x3041, 0x3096))),
        ("영어 단어", "English words", chunks(WORDS, " ")),
        ("숫자", "Digits", sample(string.digits + " ,")),
        ("ASCII 구두점", "ASCII punctuation",
         sample(string.punctuation)),
        ("공백", "Whitespace",
         chunks(["    ", "        ", "\t", "\n"], "")),
        ("이모지", "Emoji",
         sample(range_chars(0x1f600, 0x1f64f))),
        ("한국어 문단", "Korean paragraphs",
         "\n\n".join(r["ko"] for r in parallel)),
    ]


samples = probes(parallel)[:3 if QUICK else None]
d_jobs = [item("D", ko, en, text) for ko, en, text in samples]
scripts = []
measured = report_jobs("D", "문자군", d_jobs, probe_models)
for j, _, old, new in measured:
    length = len(j["text"])
    rates = [v / length if v is not None else None
             for v in (old, new)]
    scripts.append((j["item"], j["item_en"], rates))

## E. 100만 자 입력 비용

실험 A의 언어별 순 토큰 합계를 같은 언어의 총 글자 수로 나눕니다. 여기에 설정 셀의 입력 100만 토큰당 단가를 곱하면 100만 자당 예상 달러 비용이 됩니다. 표에는 지정한 환율로 환산한 원화와 달러를 함께 표시합니다. Haiku 5.5는 요청당 입력 10만 토큰 이하 구간을 가정합니다. [Anthropic 요금표](https://platform.claude.com/docs/en/about-claude/pricing)

In [ ]:
costs = []
for (lang, m), net in sums.items():
    if m in PRICE and net is not None:
        usd = PRICE[m] * net / char_totals[lang]
        krw = usd * KRW_PER_USD
        amount = f"{krw:,.2f} ({usd:.4f})"
        costs.append((lang, alias[m], amount))
emit("\nE. 입력 100만 자 요금 (Haiku 5.5는 요청당 10만 토큰 이하)")
emit(f"환율 {KRW_PER_USD:g}원/USD")
table(["언어", "모델", "원 (USD)"], costs)

## 그래프와 측정 기록

첫 그림은 한영 문단 합계와 혼합 형식별 증감률, 둘째는 문자군별 글자당 순 토큰 수입니다. 셋째는 비교 모델 모두에서 측정된 문서만 모아 실제 평균 길이와 글자당 순 토큰 수를 그립니다. 그래프 언어는 `ko`, `en`, `both` 중에서 선택합니다.

PNG와 원시 계수 JSON은 `results/`에 저장합니다. 한국어 그림에 쓰는 나눔고딕은 `cache/fonts/`에 받습니다. [Nanum Gothic · SIL Open Font License](https://github.com/google/fonts/tree/main/ofl/nanumgothic)

In [ ]:
def chart_font():
    path = CACHE / "fonts" / "NanumGothic-Regular.ttf"
    try:
        if not path.is_file():
            response = requests.get(FONT_URL, timeout=25)
            response.raise_for_status()
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(response.content)
        font_manager.fontManager.addfont(str(path))
        font = font_manager.FontProperties(fname=str(path))
        return font.get_name()
    except (OSError, RuntimeError, requests.RequestException):
        emit("경고: 나눔고딕을 사용할 수 없어 설치 글꼴을 사용합니다.")
        names = font_manager.fontManager.get_font_names()
        for name in ("AppleGothic", "Malgun Gothic",
                     "Noto Sans CJK KR"):
            if name in names:
                return name
        emit("경고: 기본 글꼴에서 한글 글리프가 누락될 수 있습니다.")
        return "DejaVu Sans"


def finish(fig, ax, filename, lang, axis):
    ax.set_axisbelow(True)
    ax.grid(axis=axis, alpha=.16)
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", "Glyph .* missing from font",
        )
        path = RESULTS / f"{filename}_{lang}.png"
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.show()
    plt.close(fig)


def nice(model, i, ko):
    """claude-haiku-4-5 -> Haiku 4.5 (구)"""
    if not model:
        return "—"
    _, fam, *ver = model.split("-")
    tag = ("구", "신")[i] if ko else ("old", "new")[i]
    return f"{fam.title()} {'.'.join(ver)} ({tag})"


def plot(lang):
    ko = lang == "ko"
    idx = 0 if ko else 1
    bars = language_bars + format_bars
    fig, ax = plt.subplots(figsize=(8, 5))
    usable = [b for b in bars if b[2] is not None]
    rects = ax.barh([b[idx] for b in usable],
                    [b[2] for b in usable], color=PALETTE[0])
    ax.bar_label(rects, fmt="%.1f", padding=3, color=PALETTE[2])
    ax.margins(x=.17)
    ax.invert_yaxis()
    ax.axvline(0, color=PALETTE[2], lw=.8)
    ax.set_xlabel("순 토큰 증감률 (%)" if ko
                  else "Change in net tokens (%)")
    finish(fig, ax, "fig1_change_by_content", lang, "x")

    fig, ax = plt.subplots(figsize=(8, 5.6))
    names = [nice(m, i, ko) for i, m in enumerate(pair)]
    rows2 = [s for s in scripts
               if any(v is not None for v in s[2])]
    for i, model in enumerate(pair):
        if model:
            ys = [k + (i - .5) * .38
                  for k in range(len(rows2))]
            rates = [s[2][i] for s in rows2]
            xs = [v if v is not None else math.nan
                  for v in rates]
            ax.barh(ys, xs, height=.36, color=PALETTE[i],
                    label=names[i])
    ax.set_yticks(range(len(rows2)),
                  [s[idx] for s in rows2])
    ax.invert_yaxis()
    if all(pair):
        ax.legend()
    tpc = "글자당 순 토큰" if ko else "Net tokens per character"
    ax.set_xlabel(tpc)
    finish(fig, ax, "fig2_tokens_per_char_by_script", lang, "x")

    fig, ax = plt.subplots(figsize=(8, 4.6))
    active = [p for p, m in enumerate(pair) if m]
    labels = ("한국어", "영어") if ko else ("Korean", "English")
    for k, language in enumerate(("ko", "en")):
        for i, model in enumerate(pair):
            points = []
            for key, rows in sorted(lengths.items()):
                doclang, limit = key
                if doclang != language or not model:
                    continue
                matched = [r for r in rows
                           if all(r[p + 1] is not None
                                  for p in active)]
                if matched:
                    n = sum(r[0] for r in matched)
                    rate = sum(r[i + 1] for r in matched) / n
                    points.append((n / len(matched), rate))
            if points:
                xs, ys = zip(*sorted(points))
                ax.plot(xs, ys, ("-o", ":s")[i],
                        color=PALETTE[k], ms=4,
                        label=f"{labels[k]}, {names[i]}")
    ax.set_xscale("log")
    ax.set_xlabel("평균 실제 글자 수" if ko
                  else "Mean actual prefix length (characters)")
    ax.set_ylabel(tpc)
    if len(ax.lines) >= 2:
        ax.legend()
    finish(fig, ax, "fig3_length_linearity", lang, "both")


RESULTS.mkdir(exist_ok=True)
records = [r for exp in ("base", "A", "B", "C", "D")
           for r in by_exp[exp]]
failures = wiki_failures.copy()
for r in records:
    if "error" in r:
        label = f"{r['experiment']} {r['item']} {r['model']}"
        failures.append(label + " " + r["error"])
for m in sorted(missing):
    emit("모델 없음, 제외 " + m)
result = dict(
    utc=stamp, sdk=anthropic.__version__, dry_run=False,
    quick=QUICK, repeats=REPEATS, models=models,
    comparison=pair,
    krw_per_usd=KRW_PER_USD, usd_per_mtok=PRICE,
    sources=sources,
    baseline=bases, missing_models=sorted(missing),
    failures=failures, records=records,
)
with (RESULTS / "results.json").open("w", encoding="utf8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
    f.write("\n")

langs = ["ko", "en"] if CHART_LANG == "both" else [CHART_LANG]
if "ko" in langs:
    plt.rcParams["font.family"] = chart_font()
plt.rcParams.update({
    "axes.unicode_minus": False, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10,
})
for lang in langs:
    plot(lang)

## 실패한 호출

실패한 입력이 있으면 아래에서 실험명·항목·오류 종류를 확인하세요. 성공한 호출의 측정값은 결과 파일에 남습니다.

In [ ]:
emit("\n실패 목록")
if failures:
    emit(f"⚠ 경고: 실패한 호출 {len(failures)}건")
    for failure in failures:
        emit(failure)
else:
    emit("실패한 호출 없음")